# 05 · ISIMIP model diagnostics

Four-GCM ensemble for LPJmL and a separate LPJmL/GEPIC/PEPIC comparison for maize, rice, soybean and wheat. Cross-model diagnostics do not replace the accepted primary LPJmL crop-reference layer.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

LPJML = RUNS/'lpjml'/'06_GLOBAL_LPJmL_pirrww_SPAM_weighted_RAW_2000_2020.csv.gz'
MULTI = RUNS/'multimodel'/'10_GLOBAL_ISIMIP2b_3MODEL_4CROP_pirrww_RAW_2000_2020.csv.gz'
OUT = RUNS/'model-qc'
KEY=['country_iso3','year','climate_scenario','fable_product_norm','lpjml_crop_code']

In [ ]:
def model_median(raw):
    required=KEY+['climate_forcing','weighted_pirrww_mm_yr','spatial_coverage_fraction']
    missing=set(required)-set(raw)
    if missing:raise ValueError(f'Missing fields: {missing}')
    if raw.duplicated(KEY+['climate_forcing']).any():raise ValueError('Duplicate model/forcing keys')
    counts=raw.groupby(KEY,dropna=False)['climate_forcing'].nunique()
    if not counts.eq(4).all():raise ValueError('Original LPJmL four-GCM ensemble is incomplete')
    out=raw.groupby(KEY,as_index=False).agg(
        n_climate_forcings=('climate_forcing','nunique'),
        median_pirrww_mm_yr=('weighted_pirrww_mm_yr','median'),
        mean_pirrww_mm_yr=('weighted_pirrww_mm_yr','mean'),
        min_spatial_coverage=('spatial_coverage_fraction','min'),
        mean_spatial_coverage=('spatial_coverage_fraction','mean'))
    out['qc_status']=np.where(out.min_spatial_coverage>=0.99,'PASS','REVIEW_COVERAGE')
    return out

In [ ]:
if not LPJML.is_file():raise FileNotFoundError(LPJML)
raw=pd.read_csv(LPJML)
median=model_median(raw)
print('LPJmL country-crop-year-pathway records:',len(median))
print(median.groupby(['year','climate_scenario']).size())
OUT.mkdir(parents=True,exist_ok=True)
path=OUT/'07_GLOBAL_LPJmL_CWR_coefficients_2000_2020.csv'
if path.exists():raise FileExistsError(path)
median.to_csv(path,index=False)

In [ ]:
if MULTI.is_file():
    multi=pd.read_csv(MULTI)
    required=['impact_model','climate_forcing','weighted_pirrww_mm_yr',
              'country_iso3','year','climate_scenario','crop_code']
    missing=set(required)-set(multi)
    if missing:raise ValueError(f'Three-model source columns missing: {missing}')
    scope=multi[multi.crop_code.isin(['mai','ric','soy','whe'])].copy()
    group=['impact_model','country_iso3','year','climate_scenario','crop_code']
    model=scope.groupby(group,as_index=False)['weighted_pirrww_mm_yr'].median()
    wide=model.pivot_table(index=['country_iso3','year','climate_scenario','crop_code'],
        columns='impact_model',values='weighted_pirrww_mm_yr').reset_index()
    path=OUT/'11_GLOBAL_ISIMIP2b_3MODEL_4CROP_MODEL_MEDIANS_2000_2020.csv'
    if path.exists():raise FileExistsError(path)
    model.to_csv(path,index=False)
    print('Three-model diagnostic (missing model values retained):',len(wide))
else:
    print('Three-model raw source absent: comparison not executed; no validation claimed')